In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import matplotlib
matplotlib.use('Agg')  # use non-interactive Agg backend to run headless
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

# Ensure directories exist
os.makedirs('figures', exist_ok=True)

# --- 1. Reload data & reproduce train/test split (same as prior steps) ---
df = pd.read_csv('data/processed/churn_engineered.csv')
df = df.select_dtypes(include='number')
X = df.drop('Churn', axis=1)
y = df['Churn']

imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_smote, y_smote, test_size=0.2, random_state=42, stratify=y_smote
)

# Reload trained model
with open('models/xgb_model.pkl', 'rb') as f:
    model = pickle.load(f)

# 2. Create SHAP explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print(f"SHAP values shape: {np.shape(shap_values)}")

# 3. Global feature importance (average impact)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('Top Features Driving Churn (SHAP)', fontsize=14)
plt.tight_layout()
plt.savefig('figures/shap_importance.png', dpi=150)
plt.close()
print("Global SHAP relevance plot saved to 'figures/shap_importance.png'")

# 4. Individual prediction explainability
# Find the first row position where the customer actually churned (y_test == 1)
churn_customer_pos = np.where(y_test.values == 1)[0][0]
expected_val = explainer.expected_value

# For binary/multi-class trees, shap_values might have an extra dimension or be a simple array
if isinstance(shap_values, list):
    # list of outputs per class
    cust_shap = shap_values[1][churn_customer_pos]
    exp_val = expected_val[1] if isinstance(expected_val, (list, np.ndarray)) else expected_val
elif len(np.shape(shap_values)) == 3:
    # shape is (n_samples, n_features, n_classes) or similar
    cust_shap = shap_values[churn_customer_pos, :, 1]
    exp_val = expected_val[1] if isinstance(expected_val, (list, np.ndarray)) else expected_val
else:
    # simple 1D output or 2D array (n_samples, n_features)
    cust_shap = shap_values[churn_customer_pos]
    exp_val = expected_val

print(f"Base value limit (expected): {exp_val}")
print(f"Customer SHAP contributions: {cust_shap[:5]}...")

# 5. Dependence plots for top features
# We will save these plots to files
plt.figure(figsize=(7, 5))
shap.dependence_plot('Contract', shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig('figures/shap_dependence_contract.png', dpi=150)
plt.close()
print("Dependence plot for Contract saved to 'figures/shap_dependence_contract.png'")

plt.figure(figsize=(7, 5))
shap.dependence_plot('MonthlyCharges', shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig('figures/shap_dependence_monthly_charges.png', dpi=150)
plt.close()
print("Dependence plot for MonthlyCharges saved to 'figures/shap_dependence_monthly_charges.png'")
print("\n✅ SHAP analysis executed successfully!")


SHAP values shape: (2070, 15)


Global SHAP relevance plot saved to 'figures/shap_importance.png'
Base value limit (expected): -0.00046569298137910664
Customer SHAP contributions: [ 1.220068    0.13503818 -0.00616747 -0.03135369  0.37347475]...
Dependence plot for Contract saved to 'figures/shap_dependence_contract.png'


Dependence plot for MonthlyCharges saved to 'figures/shap_dependence_monthly_charges.png'

✅ SHAP analysis executed successfully!
